# Factor Pricing ML — Exploration

Pipeline complet : données -> baseline OLS -> modèles ML -> comparaison hors échantillon -> backtest.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_fama_french_5factors, load_asset_returns
from models import fit_ols_baseline, fit_ml_models, evaluate_models, time_series_split
from backtest import compare_to_buy_and_hold

%matplotlib inline

## 1. Chargement des données

In [2]:
ff5 = load_fama_french_5factors()
ff5

,Mkt-RF,SMB,HML,RMW,CMA,RF
date,,,,,,
1963-07-01,-0.0039,-0.0048,-0.0084,0.0064,-0.0115,0.0027
1963-08-01,0.0508,-0.0080,0.0172,0.0040,-0.0038,0.0025
1963-09-01,-0.0157,-0.0043,-0.0002,-0.0078,0.0015,0.0027
1963-10-01,0.0254,-0.0134,-0.0003,0.0279,-0.0225,0.0029
1963-11-01,-0.0086,-0.0085,0.0178,-0.0043,0.0227,0.0027
...,...,...,...,...,...,...
2026-04-01,0.0995,0.0046,-0.0137,-0.0415,-0.0380,0.0029
2026-05-01,0.0491,-0.0265,-0.0231,-0.0816,-0.0146,0.0031
2026-06-01,-0.0107,0.0436,0.0358,-0.0507,0.0345,0.0029


In [9]:
# Actif test : S&P 500 (à remplacer par un portefeuille trié si tu veux aller plus loin)
asset = load_asset_returns(["^GSPC"], start="2000-01-01")
asset.columns = ["SP500"]
asset

[*********************100%***********************]  1 of 1 completed


,SP500
Date,
2000-02-29,-0.020108
2000-03-31,0.096720
2000-04-30,-0.030796
2000-05-31,-0.021915
2000-06-30,0.023934
...,...
2026-05-31,0.051470
2026-06-30,-0.010646
2026-07-31,-0.001285


## 2. Préparation des données (aligner les dates, calculer le rendement excédentaire)

In [13]:
ff5 = ff5.copy()
asset = asset.copy()

ff5.index = ff5.index.to_period("M")
asset.index = asset.index.to_period("M")

df = ff5.join(asset, how="inner")
df["excess_ret"] = df["SP500"] - df["RF"]
df = df.dropna()

X = df[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]]
y = df["excess_ret"]

X_train, X_test, y_train, y_test = time_series_split(X, y, test_size=0.2)
print(f"Train: {len(X_train)} obs, Test: {len(X_test)} obs")

DatetimeIndex(['1963-07-01', '1963-08-01', '1963-09-01', '1963-10-01',
               '1963-11-01', '1963-12-01', '1964-01-01', '1964-02-01',
               '1964-03-01', '1964-04-01',
               ...
               '2025-11-01', '2025-12-01', '2026-01-01', '2026-02-01',
               '2026-03-01', '2026-04-01', '2026-05-01', '2026-06-01',
               '2026-07-01', '2026-08-01'],
              dtype='datetime64[us]', name='date', length=758, freq=None)
Train: 0 obs, Test: 0 obs


### 2.1 Visualisation des données

Quelques contrôles visuels avant d'estimer les modèles : performance cumulée du S&P 500, volatilité glissante, distribution du rendement excédentaire et corrélations entre facteurs.


In [ ]:
# Performance cumulée du S&P 500
plot_dates = df.index.to_timestamp() if isinstance(df.index, pd.PeriodIndex) else df.index
sp500_wealth = (1 + df["SP500"]).cumprod()

plt.figure(figsize=(12, 5))
plt.plot(plot_dates, sp500_wealth, linewidth=1.8)
plt.title("Croissance de 1 $ investi dans le S&P 500")
plt.xlabel("Date")
plt.ylabel("Valeur du portefeuille")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Volatilité annualisée glissante sur 12 mois
rolling_vol = df["SP500"].rolling(12).std() * (12 ** 0.5)

plt.figure(figsize=(12, 4))
plt.plot(plot_dates, rolling_vol, linewidth=1.5)
plt.title("Volatilité annualisée glissante du S&P 500 — 12 mois")
plt.xlabel("Date")
plt.ylabel("Volatilité annualisée")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution du rendement excédentaire
plt.figure(figsize=(9, 4))
plt.hist(df["excess_ret"].dropna(), bins=35, edgecolor="white")
plt.axvline(df["excess_ret"].mean(), linestyle="--", linewidth=1.5,
            label=f"Moyenne = {df['excess_ret'].mean():.3%}")
plt.title("Distribution du rendement excédentaire du S&P 500")
plt.xlabel("Rendement excédentaire mensuel")
plt.ylabel("Fréquence")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Matrice de corrélation entre facteurs et rendement excédentaire
corr_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "excess_ret"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Corrélation")
ax.set_title("Corrélations — facteurs Fama-French et rendement excédentaire")
plt.tight_layout()
plt.show()


## 3. Baseline OLS

In [5]:
ols_model = fit_ols_baseline(y_train, X_train)
print(ols_model.summary())

ValueError: zero-size array to reduction operation maximum which has no identity

## 4. Modèles ML

In [ ]:
ml_models = fit_ml_models(X_train, y_train)
results = evaluate_models(ml_models, X_test, y_test, ols_model=ols_model, X_test_ols=X_test)
results

### 4.1 Visualisation des performances des modèles

Les métriques renvoyées par `evaluate_models` sont tracées automatiquement, sans supposer à l'avance leurs noms exacts.


In [ ]:
# Comparaison visuelle des métriques disponibles
model_col = next((c for c in results.columns if c.lower() == "model"), None)
numeric_metrics = [
    c for c in results.select_dtypes(include="number").columns
    if c != model_col
]

if model_col is not None and numeric_metrics:
    for metric in numeric_metrics:
        plt.figure(figsize=(8, 4))
        plt.bar(results[model_col].astype(str), results[metric])
        plt.title(f"Comparaison des modèles — {metric}")
        plt.xlabel("Modèle")
        plt.ylabel(metric)
        plt.xticks(rotation=30, ha="right")
        plt.grid(axis="y", alpha=0.2)
        plt.tight_layout()
        plt.show()
else:
    print("Aucune métrique numérique exploitable trouvée dans `results`.")


## 5. Backtest : la meilleure stratégie ML bat-elle le buy-and-hold ?

In [ ]:
best_model_name = results.iloc[0]["model"]
if best_model_name in ml_models:
    best_preds = pd.Series(ml_models[best_model_name].predict(X_test), index=X_test.index)
    summary = compare_to_buy_and_hold(best_preds, y_test)
    display(summary)
else:
    print("Le meilleur modèle est l'OLS — refais tourner avec ses prédictions si besoin.")

### 5.1 Réel vs prédit sur l'échantillon test

On visualise la dynamique temporelle des prédictions du meilleur modèle ML, puis leur proximité avec les rendements observés.


In [ ]:
if best_model_name in ml_models:
    pred_dates = X_test.index.to_timestamp() if isinstance(X_test.index, pd.PeriodIndex) else X_test.index

    plt.figure(figsize=(12, 5))
    plt.plot(pred_dates, y_test.values, label="Réel", linewidth=1.7)
    plt.plot(pred_dates, best_preds.values, label=f"Prédit — {best_model_name}", linewidth=1.4)
    plt.axhline(0, linewidth=0.8)
    plt.title("Rendement excédentaire : réel vs prédit")
    plt.xlabel("Date")
    plt.ylabel("Rendement excédentaire")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test.values, best_preds.values, alpha=0.65)
    lo = min(y_test.min(), best_preds.min())
    hi = max(y_test.max(), best_preds.max())
    plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.2)
    plt.title(f"Réel vs prédit — {best_model_name}")
    plt.xlabel("Rendement réel")
    plt.ylabel("Rendement prédit")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


## Prochaines étapes

- Étendre à d'autres actifs / portefeuilles triés (taille, valeur)
- Ajouter du feature engineering (rendements décalés, volatilité réalisée)
- Analyse d'importance des facteurs (SHAP) pour interpréter les modèles ML
- Tester la robustesse sur différentes périodes (crise 2008, COVID, etc.)